# NaN-loss diagnosis

When all-NaN losses appear during joint YC + futures training, the
culprit is almost always one of these (in decreasing order of
probability):

1. **BondNet output saturating** — random init produces extreme bond
   prices, the per-snapshot CTD min is huge or near zero, the gradient
   blows up, parameters become inf/NaN, the next forward pass is all
   NaN.
2. **NSDE drift blowing up** — at init, `f(t, z)` can grow without
   bound; large `dt` × large drift makes `z` explode within a few
   solver steps, the decoder produces huge `r`, `exp(-∫r ds)` underflows
   to zero, `log(P)` returns `-inf`, the yield loss is `+inf` × infinity = NaN.
3. **`r0` shift mismatch** — only an issue when the rate convention is
   wrong (the percent→decimal fix from math_review §1 closes this).
4. **Optimiser step on NaN gradients** — once one parameter goes NaN,
   every forward / loss is NaN forever. `grad_clip_norm` only protects
   against finite-but-large grads, not against NaN.

This notebook walks through each layer of the forward pass on a
freshly-initialised model and **localises** which stage first
produces a non-finite value. Run cells top-to-bottom.

In [ ]:
# Bootstrap repo root onto sys.path
import sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import torch, math
torch.manual_seed(0)
print('repo =', ROOT)


## 1. Build a small but representative joint setup

In [ ]:
from datetime import datetime, timedelta
from src.configs import (
    DataLoaderCfg, EncoderCfg, NSDECfg, SimpleBondNetCfg, TrainerCfg,
)
from src.dataloaders import MarketDataLoader
from src.models.short_rate_model import ShortRateModel
from src.training.trainer import Trainer

BOND_FEAT_DIM = 8

dl = MarketDataLoader(DataLoaderCfg(
    data_path=str(ROOT / 'data2'),
    start_date=datetime(2021, 1, 1),
    end_date=datetime(2022, 12, 31),
    max_maturity=5,
    enable_yield=True, enable_short_rate=True, enable_futures=True,
))

encoder_cfg = EncoderCfg(mode='simple')
encoder_cfg.net = {'type': 'lstm', 'n_layers': 2, 'n_units': 64, 'dropout': 0.1, 'bidirectional': True}
encoder_cfg.out_norm = 'layernorm'

nsde_cfg = NSDECfg(type='simple', noise_type='diagonal')
nsde_cfg.solver = 'custom_euler'
nsde_cfg.dt = 1 / 64
nsde_cfg.drift     = {'type': 'mlp', 'n_layers': 2, 'n_units': [64, 64], 'activation': 'gelu'}
nsde_cfg.diffusion = {'type': 'mlp', 'n_layers': 2, 'n_units': [32, 32], 'activation': 'gelu', 'out_activation': 'softplus'}

bondnet_cfg = SimpleBondNetCfg(
    latent_dim=16, bond_feat_dim=BOND_FEAT_DIM,
    latent_n_layers=1, latent_n_units=32,
    bond_n_layers=1,   bond_n_units=16,
    fusion_n_layers=1, fusion_n_units=32,
    activation='silu', output_positive=True,
)

model = ShortRateModel(
    name='nan_diag', encoder=encoder_cfg, nsde=nsde_cfg,
    bondnet=bondnet_cfg, latent_dim=16,
)
tcfg = TrainerCfg()
tcfg.n_paths = 32; tcfg.batch_window = 4; tcfg.window_step = 1
tcfg.lookback = 16; tcfg.lookback_freq = 1; tcfg.dt = 1 / 64
tcfg.use_amp = False; tcfg.early_stopping.enabled = False
trainer = Trainer(model=model, dataloader=dl, config=tcfg, device='cpu')

test_date = dl.calendar.dates[200]
print('test date =', test_date.date())


## 2. Per-stage finite check helper

In [ ]:
def stage(name, t):
    """Print finite-stats for a tensor."""
    if not isinstance(t, torch.Tensor):
        print(f'{name:35s}  -> not a tensor: {type(t).__name__}')
        return
    n_nan = torch.isnan(t).sum().item()
    n_inf = torch.isinf(t).sum().item()
    finite = t[torch.isfinite(t)]
    if finite.numel():
        lo, hi, mu = finite.min().item(), finite.max().item(), finite.mean().item()
    else:
        lo = hi = mu = float('nan')
    flag = '!! NaN/Inf' if (n_nan + n_inf) else ''
    print(f'{name:35s}  shape={tuple(t.shape)}  '
          f'min={lo:+.3e}  max={hi:+.3e}  mean={mu:+.3e}  '
          f'nan={n_nan}  inf={n_inf}  {flag}')


## 3. Walk one forward step, layer by layer

In [ ]:
from src.finance.pricer_v2 import to_year_fraction

snap = trainer._get_snapshot(test_date)
ts   = trainer._make_ts(snap)
stage('ts  (simulation grid)', ts)

# Encoder
past = trainer._get_history(test_date)
z_t  = model.encode(past)
stage('z_t  (encoder output)', z_t)

# NSDE forward
latent_paths = model.simulate(z_t, n_paths=tcfg.n_paths, ts=ts, decode=False)
stage('latent_paths', latent_paths)

# Decoder + r0 shift
r0 = trainer._get_r0(test_date)
stage('r0', r0)
realisations = trainer._decode(latent_paths, r0=r0)
stage('realisations  (short-rate)', realisations)

# Yield-curve branch
P = trainer.pricer.price_zcb(realisations, snap.yield_curve.maturities)
stage('P  (ZCB prices)', P)
y = trainer.pricer.price_yield_curve(realisations, snap.yield_curve.maturities)
stage('y  (model yields, decimal)', y)

# Futures branch
if snap.futures is not None and model.bondnet is not None:
    fut_target = snap.futures
    bf = snap.bonds_metadata.features
    stage('bond_features', bf)
    dlv_years = to_year_fraction(fut_target.delivery_dates, fut_target.asof_date)
    stage('dlv_years', dlv_years)
    fut_prices = trainer.pricer.price_futures(
        bondnet=model.bondnet, bond_features=bf,
        latent_paths=latent_paths, simulated_times=ts, target=fut_target,
    )
    stage('model futures prices', fut_prices)
    stage('market futures prices', fut_target.prices)
    stage('|model - market| futures', (fut_prices - fut_target.prices).abs())

# The actual loss the trainer would form
day_loss, comps = trainer._get_loss(realisation=realisations,
                                     snapshot=snap,
                                     latent_repr=latent_paths,
                                     ts=ts)
stage('day_loss', day_loss)
print('components:', {k: f'{v:.4g}' for k, v in comps.items()})


## 4. Read the diagnosis

Look at the **first row** above whose `nan` or `inf` count is non-zero.
Every later stage that depends on it will inherit the NaN.

| Stage where NaN first appears | Likely root cause | First fix to try |
|---|---|---|
| `z_t` (encoder output)         | Encoder lookback contains NaN inputs (e.g. unfilled short-rate column). | Tighten `start_date` so the lookback fits inside the canonical calendar. |
| `latent_paths`                  | NSDE drift / diffusion exploded inside the Euler loop. | Reduce `nsde.dt`; cap drift with `out_activation='tanh'` on a small scaled layer; pre-train encoder + NSDE on yields only first. |
| `realisations`                  | Decoder output went non-finite even with `latent_paths` finite — usually `decoder(huge_z)` overflowing. | Add `out_norm='layernorm'` to encoder (likely already on); reduce learning rate; or cap `latent_paths` with `LayerNorm` on the trunk. |
| `P`                             | `cumsum(realisations) * dt` overflowed. Almost always means the rate convention is wrong (percent vs decimal). | Confirm `market y` < ~0.2 — if it's > 5, the loader didn't divide by 100 (math_review §1). |
| `y` only                        | `log(P=0)` because integral exploded but didn't overflow yet. | Same as above. |
| `model futures prices`          | BondNet input is finite but the network saturated. | Pre-train BondNet on synthetic / analytical bond prices; or run yield-only first then turn on futures. |
| `day_loss` only                 | `MSE(inf, finite)` or `MSE(finite, inf)`. Inspect both sides. | Trace which `comps` entry exploded. |


## 5. Quick mitigation experiments

Three independent levers, applied to the *same* random init:

1. **Drop `lr` 10×** — kills runaway grad updates.
2. **Disable futures branch (`enable_futures=False`)** — isolates whether the NSDE / decoder paths alone are stable.
3. **Lower `nsde.dt`** — smaller per-step drift × dt, less chance of blow-up.

In [ ]:
import copy

def is_finite(t):
    return torch.isfinite(t).all().item()

def quick_train_step(trainer, dates):
    """Run one window and report whether the loss is finite, plus a backward pass."""
    floss, lt, n_ok = trainer._train_one_window(dates)
    print(f'  window-loss = {floss:.4g}, n_dates_ok = {n_ok}, requires_grad={lt.requires_grad}')
    if not torch.isfinite(lt):
        print('  → NaN/Inf — cannot backward.')
        return False
    if lt.requires_grad:
        try:
            lt.backward()
            param_finite = all(p.grad is None or is_finite(p.grad) for p in trainer.model.parameters())
            print(f'  → backward OK,  all grads finite: {param_finite}')
            return param_finite
        except Exception as e:
            print(f'  → backward FAILED: {e!r}')
            return False
    print('  → no grad — loss not differentiable.')
    return False

batch = list(dl.calendar.dates[200:204])

# Baseline
print('Baseline (joint, lr=1e-3):')
quick_train_step(trainer, batch)


In [ ]:
# Mitigation A — drop the learning rate by 10x (rebuild optimiser)
torch.manual_seed(0)
model_a = ShortRateModel(
    name='nan_diag_A', encoder=encoder_cfg, nsde=nsde_cfg,
    bondnet=bondnet_cfg, latent_dim=16,
)
tcfg_a = copy.deepcopy(tcfg)
tcfg_a.optimizer.params = {'lr': 1e-4, 'weight_decay': 1e-4}
trainer_a = Trainer(model=model_a, dataloader=dl, config=tcfg_a, device='cpu')
print('Mitigation A — lr 1e-4 (10x smaller):')
quick_train_step(trainer_a, batch)


In [ ]:
# Mitigation B — yields only (no futures); isolates whether the NSDE/decoder paths alone are stable
dl_yc = MarketDataLoader(DataLoaderCfg(
    data_path=str(ROOT / 'data2'),
    start_date=datetime(2021, 1, 1), end_date=datetime(2022, 12, 31),
    max_maturity=5,
    enable_yield=True, enable_short_rate=True, enable_futures=False,
))
torch.manual_seed(0)
model_b = ShortRateModel(
    name='nan_diag_B', encoder=encoder_cfg, nsde=nsde_cfg, latent_dim=16,
)  # no bondnet
trainer_b = Trainer(model=model_b, dataloader=dl_yc, config=tcfg, device='cpu')
print('Mitigation B — yields-only (no BondNet):')
quick_train_step(trainer_b, batch)


In [ ]:
# Mitigation C — finer NSDE dt (smaller drift × dt steps)
torch.manual_seed(0)
nsde_cfg_c = copy.deepcopy(nsde_cfg)
nsde_cfg_c.dt = 1 / 256
model_c = ShortRateModel(
    name='nan_diag_C', encoder=encoder_cfg, nsde=nsde_cfg_c,
    bondnet=bondnet_cfg, latent_dim=16,
)
trainer_c = Trainer(model=model_c, dataloader=dl, config=tcfg, device='cpu')
print('Mitigation C — nsde.dt = 1/256 (4x finer):')
quick_train_step(trainer_c, batch)


## 6. Action plan

Translate the section-3 diagnosis + section-5 mitigation table into a concrete order of operations:

1. **If yields-only run is stable but joint isn't** → the failure is on the BondNet / futures branch. Two low-risk fixes:
    - Add a **futures-warmup**: train yields-only for ~5 epochs, then enable futures.
    - Or expose **per-target loss weights** (`λ_y`, `λ_f`) on `TrainerCfg` and start with `λ_f = 0.01` (this is also priority P1 in `optimization_plan.md` §10.1).

2. **If yields-only is also broken** → the failure is in the NSDE/decoder. Try in order:
    - Smaller `lr` (1e-4 → 5e-5).
    - Smaller `nsde.dt` (1/256 or 1/512).
    - Re-check rate convention: `model y` for a flat path should sit at the same scale as the market yields (~0.05 in decimal). If it's at 5, the percent→decimal fix didn't take effect.
    - Add an output norm to the LATENT branch (LayerNorm on `z_t` already on by default; check `cfg.out_norm == 'layernorm'`).

3. **Always**: ensure `grad_clip_norm` is on (1.0 is the project default).

4. **Once a working baseline exists**, gradually re-introduce the broken-but-desirable features (larger batch, higher LR, futures branch) one at a time, monitoring `pricer.last_bond_stats` and the per-target loss components each epoch.